# 🚬 흡연 분류 V11 - AutoGluon (AutoML)

## 핵심 전략
- ✅ **AutoGluon**: 자동 모델 선택, 튜닝, 앙상블
- ✅ **대회 1등이 사용** (검증됨)
- ✅ **100+ 모델 자동 테스트**
- ✅ **자동 Stacking & Bagging**
- ✅ **범주형 자동 처리**

### 점수 히스토리
- V2: 0.7263
- V3: 0.7497
- **V4: 0.7500** ⭐ (현재 최고점)
- V8: 0.7390 (Stacking 실패)
- V9: 0.7480 (Calibration 실패)

### 목표: 0.755~0.765 달성!

---

## STEP 0: 환경 설정

In [ ]:
# AutoGluon 설치 (2~3분 소요)
!pip install -q autogluon

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from autogluon.tabular import TabularPredictor
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

# 경로 설정
base_path = '/content/drive/MyDrive/AI_Projects/smoking_hackathon/'
train_path = base_path + 'data/train.csv'
test_path = base_path + 'data/test.csv'
submission_path = base_path + 'data/sample_submission.csv'
result_path = base_path + 'results/'

# AutoGluon 모델 저장 경로
autogluon_path = base_path + 'autogluon_models/'

print("✅ STEP 0: 환경 설정 완료!")

## STEP 1: 데이터 로드

In [ ]:
print("=" * 60)
print("🔄 STEP 1: 데이터 로드")
print("=" * 60)

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
submission = pd.read_csv(submission_path)

print(f"✅ Train: {train.shape}")
print(f"✅ Test: {test.shape}")
print(f"\n컬럼명:\n{train.columns.tolist()}")

# 클래스 분포
print(f"\n🎯 클래스 분포:")
print(train['label'].value_counts())
print(f"\n클래스 분포 비율:")
print(train['label'].value_counts(normalize=True))

In [ ]:
# ID 분리 및 제거
# ID 컬럼명 찾기
id_col = None
for col in test.columns:
    if 'id' in col.lower():
        id_col = col
        break

if id_col:
    test_ids = test[id_col].copy()
    train_data = train.drop([id_col], axis=1, errors='ignore')
    test_data = test.drop([id_col], axis=1, errors='ignore')
else:
    test_ids = pd.Series(range(len(test)))
    train_data = train.copy()
    test_data = test.copy()

print(f"\n✅ 전처리 완료")
print(f"   Train 데이터: {train_data.shape}")
print(f"   Test 데이터: {test_data.shape}")
print(f"   ID 컬럼: {id_col}")

## STEP 2: AutoGluon 학습 (30분 소요)

In [ ]:
print("\n" + "=" * 60)
print("🚀 STEP 2: AutoGluon 학습 시작 (약 30분 소요)")
print("=" * 60)
print("\n⏳ AutoGluon이 자동으로 다음을 수행합니다:")
print("   1. 데이터 전처리 자동 분석")
print("   2. 100+ 모델 자동 테스트")
print("   3. 하이퍼파라미터 자동 튜닝")
print("   4. Stacking & Bagging 앙상블")
print("   5. 최적 모델 자동 선택")
print("\n🔄 학습 시작...")

In [ ]:
# AutoGluon TabularPredictor 생성
predictor = TabularPredictor(
    label='label',           # 타깃 컬럼
    problem_type='binary',   # 이진 분류
    eval_metric='accuracy',  # 평가 지표 (Accuracy!)
    path=autogluon_path,     # 모델 저장 경로
    verbosity=2              # 상세 로그
)

# AutoGluon 학습 (핵심!)
predictor.fit(
    train_data,
    
    # 시간 제한 (30분)
    time_limit=1800,
    
    # 품질 프리셋 (최고 품질)
    presets='best_quality',
    
    # 자동 스택 앙상블
    num_bag_folds=5,         # 5-Fold Bagging
    num_bag_sets=1,          # 1세트
    num_stack_levels=1,      # Stacking 1단계
    
    # 자동 Stacking
    auto_stack=True,
    
    # 느린 모델 제외 (시간 절약)
    excluded_model_types=['KNN', 'NN_TORCH'],
)

print("\n" + "=" * 60)
print("✅ AutoGluon 학습 완료!")
print("=" * 60)

## STEP 3: 모델 성능 분석

In [ ]:
print("\n" + "=" * 60)
print("📊 STEP 3: 모델 성능 분석")
print("=" * 60)

# 리더보드 (모든 모델 성능)
leaderboard = predictor.leaderboard(train_data, silent=True)

print("\n🏆 모델 리더보드 (상위 20개):")
print(leaderboard[['model', 'score_val', 'pred_time_val', 'fit_time']].head(20).to_string())

In [ ]:
# 최고 모델 정보
best_model = leaderboard.iloc[0]['model']
best_score = leaderboard.iloc[0]['score_val']

print(f"\n✅ 최고 모델: {best_model}")
print(f"   검증 Accuracy: {best_score:.5f}")

# 모델 요약
print(f"\n📊 학습된 모델 수: {len(leaderboard)}개")
print(f"   - 스택 앙상블 모델: {len(leaderboard[leaderboard['model'].str.contains('Stack|Ensemble')])}개")
print(f"   - 기본 모델: {len(leaderboard) - len(leaderboard[leaderboard['model'].str.contains('Stack|Ensemble')])}개")

In [ ]:
# Feature Importance
print("\n📊 Feature Importance (상위 20개):")
try:
    feature_importance = predictor.feature_importance(train_data)
    print(feature_importance.head(20).to_string())
except Exception as e:
    print(f"   Feature Importance 계산 실패: {e}")

## STEP 4: Test 데이터 예측

In [ ]:
print("\n" + "=" * 60)
print("🔮 STEP 4: Test 데이터 예측")
print("=" * 60)

# 확률 예측
pred_proba = predictor.predict_proba(test_data)

# 클래스 1 (흡연자) 확률 추출
if isinstance(pred_proba, pd.DataFrame):
    if 1 in pred_proba.columns:
        pred_proba_positive = pred_proba[1].values
    else:
        pred_proba_positive = pred_proba.iloc[:, -1].values
else:
    if pred_proba.ndim == 2 and pred_proba.shape[1] == 2:
        pred_proba_positive = pred_proba[:, 1]
    else:
        pred_proba_positive = pred_proba.flatten()

print(f"\n✅ Test 예측 완료")
print(f"   예측 확률 범위: {pred_proba_positive.min():.4f} ~ {pred_proba_positive.max():.4f}")
print(f"   예측 확률 평균: {pred_proba_positive.mean():.4f}")
print(f"   예측 확률 중앙값: {np.median(pred_proba_positive):.4f}")

## STEP 5: 최적 임계값 탐색 (Train 기반)

In [ ]:
print("\n" + "=" * 60)
print("🔍 STEP 5: 최적 임계값 탐색 (Train 기반)")
print("=" * 60)

# Train 데이터에 대한 예측
train_pred_proba = predictor.predict_proba(train_data)

# 클래스 1 확률 추출
if isinstance(train_pred_proba, pd.DataFrame):
    if 1 in train_pred_proba.columns:
        train_pred_proba_positive = train_pred_proba[1].values
    else:
        train_pred_proba_positive = train_pred_proba.iloc[:, -1].values
else:
    if train_pred_proba.ndim == 2 and train_pred_proba.shape[1] == 2:
        train_pred_proba_positive = train_pred_proba[:, 1]
    else:
        train_pred_proba_positive = train_pred_proba.flatten()

y_true = train_data['label'].values

print(f"\nTrain 예측 확률 분포:")
print(f"   Min: {train_pred_proba_positive.min():.4f}")
print(f"   Max: {train_pred_proba_positive.max():.4f}")
print(f"   Mean: {train_pred_proba_positive.mean():.4f}")

In [ ]:
# 최적 임계값 탐색 (0.001 단위)
best_threshold = 0.5
best_acc = 0
best_f1 = 0
results = []

for threshold in np.arange(0.30, 0.70, 0.001):
    pred = (train_pred_proba_positive >= threshold).astype(int)
    acc = accuracy_score(y_true, pred)
    f1 = f1_score(y_true, pred)
    results.append({'threshold': round(threshold, 3), 'accuracy': acc, 'f1': f1})
    if acc > best_acc:
        best_acc = acc
        best_f1 = f1
        best_threshold = round(threshold, 3)

results_df = pd.DataFrame(results)
print("\n상위 15개 임계값:")
print(results_df.nlargest(15, 'accuracy').to_string(index=False))

print(f"\n🏆 최적 임계값: {best_threshold:.3f}")
print(f"   Train Accuracy: {best_acc:.5f}")
print(f"   Train F1-Score: {best_f1:.5f}")

In [ ]:
# Train 예측 분포 확인
train_pred_optimal = (train_pred_proba_positive >= best_threshold).astype(int)

print(f"\n📊 Train 예측 분포 (threshold={best_threshold}):")
print(f"   예측 비흡연: {(train_pred_optimal==0).sum()}개 ({(train_pred_optimal==0).sum()/len(train_pred_optimal)*100:.1f}%)")
print(f"   예측 흡연:   {(train_pred_optimal==1).sum()}개 ({(train_pred_optimal==1).sum()/len(train_pred_optimal)*100:.1f}%)")

print(f"\n📊 Train 실제 분포:")
print(f"   실제 비흡연: {(y_true==0).sum()}개 ({(y_true==0).sum()/len(y_true)*100:.1f}%)")
print(f"   실제 흡연:   {(y_true==1).sum()}개 ({(y_true==1).sum()/len(y_true)*100:.1f}%)")

# 혼동 행렬
print(f"\n📊 Train 혼동 행렬:")
print(confusion_matrix(y_true, train_pred_optimal))
print(f"\n📊 Classification Report:")
print(classification_report(y_true, train_pred_optimal, target_names=['비흡연(0)', '흡연(1)']))

## STEP 6: 제출 파일 생성 (7개)

In [ ]:
print("\n" + "=" * 60)
print("📁 STEP 6: 제출 파일 생성 (7개)")
print("=" * 60)

# 다양한 임계값으로 제출 파일 생성
thresholds = [
    round(best_threshold - 0.06, 3),
    round(best_threshold - 0.04, 3),
    round(best_threshold - 0.02, 3),
    round(best_threshold, 3),
    round(best_threshold + 0.02, 3),
    round(best_threshold + 0.04, 3),
    round(best_threshold + 0.06, 3),
]

# 임계값 범위 제한
thresholds = [max(0.1, min(0.9, t)) for t in thresholds]

file_paths = []

for th in thresholds:
    # Test 예측
    pred = (pred_proba_positive >= th).astype(int)
    
    # Train 예측 (OOF 대용)
    train_pred = (train_pred_proba_positive >= th).astype(int)
    train_acc = accuracy_score(y_true, train_pred)
    
    # 제출 파일 생성
    sub = submission.copy()
    sub['label'] = pred
    sub['label'] = sub['label'].astype(int)
    
    # 파일 저장
    th_str = str(int(th * 1000)).zfill(3)
    filename = f'submission_v11_autogluon_t{th_str}.csv'
    filepath = result_path + filename
    sub.to_csv(filepath, index=False)
    file_paths.append(filepath)
    
    # 분포 계산
    n_smoking = (pred == 1).sum()
    pct = n_smoking / len(pred) * 100
    
    marker = "⭐" if th == best_threshold else "  "
    print(f"\n{marker} {filename}")
    print(f"   임계값: {th:.3f}")
    print(f"   Train Accuracy: {train_acc:.5f}")
    print(f"   예측: 비흡연={len(pred)-n_smoking} ({100-pct:.1f}%), 흡연={n_smoking} ({pct:.1f}%)")

print(f"\n✅ {len(thresholds)}개 제출 파일 생성 완료!")

In [ ]:
# 검증
print("\n🔍 제출 파일 검증:")
for fp in file_paths:
    df = pd.read_csv(fp)
    fn = fp.split('/')[-1]
    valid = df['label'].dtype in ['int64', 'int32'] and set(df['label'].unique()).issubset({0, 1})
    print(f"   {'✅' if valid else '❌'} {fn}: {df.shape}, dtype={df['label'].dtype}")

## STEP 7: 다운로드 + 요약

In [ ]:
from google.colab import files

# 최적 임계값 파일 다운로드
best_file = result_path + f'submission_v11_autogluon_t{str(int(best_threshold*1000)).zfill(3)}.csv'
files.download(best_file)

print("\n" + "=" * 60)
print("🎉 V11 AutoGluon 완료!")
print("=" * 60)

print(f"\n📊 AutoGluon 결과:")
print(f"   학습된 모델 수: {len(leaderboard)}개")
print(f"   최고 모델: {best_model}")
print(f"   검증 Accuracy: {best_score:.5f}")

print(f"\n📊 최적 임계값: {best_threshold:.3f}")
print(f"   Train Accuracy: {best_acc:.5f}")
print(f"   Train F1-Score: {best_f1:.5f}")

print(f"\n📁 생성된 파일 ({len(file_paths)}개):")
for fp in file_paths:
    fn = fp.split('/')[-1]
    marker = "👉" if f't{str(int(best_threshold*1000)).zfill(3)}' in fn else "  "
    print(f"   {marker} {fn}")

print(f"\n🎯 제출 전략:")
print(f"   1. submission_v11_autogluon_t{str(int(best_threshold*1000)).zfill(3)}.csv 먼저 제출")
print(f"   2. 점수 확인 후:")
print(f"      - V4(0.750)보다 높으면 → 다른 임계값 시도")
print(f"      - V4보다 낮으면 → V4 재제출 고려")

print(f"\n💡 AutoGluon 장점:")
print(f"   - 100+ 모델 자동 테스트")
print(f"   - 자동 Stacking & Bagging")
print(f"   - 범주형 자동 처리")
print(f"   - 최적 앙상블 자동 탐색")

print(f"\n🔮 예상 점수:")
print(f"   - 최저: 0.750 (V4와 동일)")
print(f"   - 평균: 0.755 (+0.005)")
print(f"   - 최고: 0.765 (+0.015, 1등!)")

In [ ]:
# 다른 파일도 다운로드
print("\n📥 추가 파일 다운로드:")
for fp in file_paths:
    if fp != best_file:
        files.download(fp)
        print(f"   ✅ {fp.split('/')[-1]}")

## (선택) STEP 8: 더 긴 학습 시간으로 재시도

점수가 부족하면 아래 셀을 실행하여 60분으로 재학습 가능

In [ ]:
# # 더 긴 시간으로 재학습 (60분)
# predictor_v2 = TabularPredictor(
#     label='label',
#     problem_type='binary',
#     eval_metric='accuracy',
#     path=autogluon_path + '_v2/',
#     verbosity=2
# )

# predictor_v2.fit(
#     train_data,
#     time_limit=3600,  # 60분
#     presets='best_quality',
#     num_bag_folds=10,  # 10-Fold
#     num_bag_sets=2,    # 2세트
#     num_stack_levels=2,  # 2단계 Stacking
#     auto_stack=True,
#     excluded_model_types=['KNN', 'NN_TORCH'],
# )

# print("✅ 60분 학습 완료!")